[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/44_sgd_momentum_solution.ipynb)

# 🟡 Solution: SGD with Momentum

Reference implementation matching the standard PyTorch SGD update without dampening or Nesterov momentum.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

class MySGD:
    def __init__(self, params, lr=0.01, momentum=0.0, weight_decay=0.0):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.weight_decay = weight_decay
        self.buffers = [torch.zeros_like(p) for p in self.params]

    def step(self):
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue

                grad = p.grad
                if self.weight_decay != 0.0:
                    grad = grad + self.weight_decay * p

                if self.momentum != 0.0:
                    self.buffers[i].mul_(self.momentum).add_(grad)
                    update = self.buffers[i]
                else:
                    update = grad

                p.add_(update, alpha=-self.lr)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


In [ ]:
# Verify
p = torch.tensor([1.0, -2.0], requires_grad=True)
opt = MySGD([p], lr=0.1, momentum=0.9)
for step in range(3):
    loss = p.pow(2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f"step={step} p={p.detach().tolist()} loss={loss.item():.4f}")


In [ ]:
# Run judge
from torch_judge import check
check('sgd_momentum')
